# DistilBERT Intent Classifier — Real Data Retraining
### Sunlytics CRS — M3 Implementation

**Purpose:** Retrain the DistilBERT 8-class intent classifier using a mixed dataset of real SIMMC 2.1 conversations and synthetic data.

**Training data:** `v5_train_mixed.csv` — 24,000 rows (3,000 per label)  
- 14,714 rows (61.3%) from SIMMC 2.1 real fashion dialogues  
- 9,286 rows (38.7%) from synthetic data (only for EXPLANATION_WHY and CHITCHAT which don't exist in SIMMC)

**Validation data:** `real_val_simmc.csv` — 2,989 rows from SIMMC 2.1 dev split  
**Test data:** `real_test_simmc.csv` — 5,483 rows from SIMMC 2.1 devtest split

---

| ID | Label | Strategy | Train Count |
|----|-------|----------|-------------|
| 0 | INITIAL_REQUEST | FULL | 3,000 |
| 1 | REFINEMENT | FULL | 3,000 |
| 2 | ATTRIBUTE_QUESTION | PARTIAL | 3,000 |
| 3 | EXPLANATION_WHY | PARTIAL | 3,000 |
| 4 | COMPARISON | PARTIAL | 3,000 |
| 5 | SELECTION_REFERENCE | PARTIAL | 3,000 |
| 6 | FEEDBACK | NO | 3,000 |
| 7 | CHITCHAT | NO | 3,000 |
| | **TOTAL** | | **24,000** |

**Expected runtime:** ~25-35 min on Colab T4 GPU

## STEP 1 — Verify GPU and Install Dependencies

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
!pip install transformers>=4.40.0 datasets scikit-learn pandas numpy matplotlib seaborn tqdm accelerate -q

## STEP 2 — Upload Data Files

Upload these 3 files from your local machine:
- `v5_train_mixed.csv` — balanced mixed training set (24,000 rows)
- `real_val_simmc.csv` — real SIMMC validation set (2,989 rows)
- `real_test_simmc.csv` — real SIMMC test set (5,483 rows)

All three files are in: `m3_implementation/adaptive_rag/distilbert_training/data/`

In [ ]:
from google.colab import files
import os, shutil

os.makedirs('data', exist_ok=True)
os.makedirs('outputs/best_model', exist_ok=True)
os.makedirs('outputs/results', exist_ok=True)

print('Upload: v5_train_mixed.csv, real_val_simmc.csv, real_test_simmc.csv')
uploaded = files.upload()

for fname in uploaded:
    shutil.move(fname, f'data/{fname}')
    print(f'  Moved {fname} -> data/{fname}')

print('\nFiles in data/:', os.listdir('data/'))

## STEP 3 — Configuration

In [ ]:
# ============================================================
# CONFIG — Real data retraining with v5 mixed dataset
# ============================================================

TRAIN_FILE = 'data/v5_train_mixed.csv'
VAL_FILE   = 'data/real_val_simmc.csv'
TEST_FILE  = 'data/real_test_simmc.csv'

MODEL_SAVE_DIR = 'outputs/best_model'
RESULTS_DIR    = 'outputs/results'

PRETRAINED_MODEL = 'distilbert-base-uncased'

LABEL_NAMES = [
    'INITIAL_REQUEST',      # 0 -> FULL
    'REFINEMENT',           # 1 -> FULL
    'ATTRIBUTE_QUESTION',   # 2 -> PARTIAL
    'EXPLANATION_WHY',      # 3 -> PARTIAL
    'COMPARISON',           # 4 -> PARTIAL
    'SELECTION_REFERENCE',  # 5 -> PARTIAL
    'FEEDBACK',             # 6 -> NO
    'CHITCHAT',             # 7 -> NO
]
NUM_LABELS = len(LABEL_NAMES)

RETRIEVAL_STRATEGY_MAP = {
    0: 'FULL', 1: 'FULL',
    2: 'PARTIAL', 3: 'PARTIAL', 4: 'PARTIAL', 5: 'PARTIAL',
    6: 'NO', 7: 'NO',
}

MAX_LEN    = 256
BATCH_SIZE = 32

# Hyperparameters
# Slightly higher LR than synthetic training (2e-5 vs 1e-5) because
# the dataset is smaller (24k vs 52k) and real data provides cleaner signal
LEARNING_RATE   = 2e-5
LABEL_SMOOTHING = 0.1   # prevents overconfidence, improves generalisation
DROPOUT         = 0.3   # regularisation
FREEZE_LAYERS   = 2     # freeze first 2 of 6 layers — allow more adaptation to real data

NUM_EPOCHS               = 10
WEIGHT_DECAY             = 0.01
WARMUP_RATIO             = 0.1
EARLY_STOPPING_PATIENCE  = 3
SEED                     = 42

print('Configuration loaded.')
print(f'  Train : {TRAIN_FILE}')
print(f'  Val   : {VAL_FILE}')
print(f'  Test  : {TEST_FILE}')
print(f'  LR    : {LEARNING_RATE}')
print(f'  Frozen layers : first {FREEZE_LAYERS} of 6')

## STEP 4 — Load and Verify Datasets

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

df_train = pd.read_csv(TRAIN_FILE)
df_val   = pd.read_csv(VAL_FILE)
df_test  = pd.read_csv(TEST_FILE)

# Fill missing label integers from label_name if needed
label_name_to_id = {n: i for i, n in enumerate(LABEL_NAMES)}
for df in [df_train, df_val, df_test]:
    if 'label' not in df.columns or df['label'].isnull().any():
        df['label'] = df['label_name'].map(label_name_to_id)

print('\n=== Dataset sizes ===')
print(f'  Train : {len(df_train):,} rows')
print(f'  Val   : {len(df_val):,} rows')
print(f'  Test  : {len(df_test):,} rows')

print('\n=== Train label distribution (should be 3000 each) ===')
train_counts = Counter(df_train['label_name'].tolist())
for name in LABEL_NAMES:
    n   = train_counts.get(name, 0)
    bar = '#' * (n // 75)
    print(f'  {name:<25} {n:>5}  {bar}')

print('\n=== Val label distribution ===')
val_counts = Counter(df_val['label_name'].tolist())
for name in LABEL_NAMES:
    n = val_counts.get(name, 0)
    print(f'  {name:<25} {n:>5}')

# Class weights for loss function (handles slight val imbalance)
label_counts = Counter(df_train['label'].tolist())
total        = len(df_train)
class_weights = torch.tensor(
    [total / (NUM_LABELS * label_counts[i]) for i in range(NUM_LABELS)],
    dtype=torch.float,
).to(device)
print(f'\nClass weights: {[round(w.item(), 3) for w in class_weights]}')

## STEP 5 — Dataset Class

In [ ]:
class IntentDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts  = df['input_text'].fillna('').tolist()
        self.labels = df['label'].tolist()
        self.tok    = tokenizer
        assert all(0 <= l < NUM_LABELS for l in self.labels), \
            f'Label out of range 0-{NUM_LABELS-1}'

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=MAX_LEN,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

print('IntentDataset class defined.')

## STEP 6 — Load Tokenizer and Model

In [ ]:
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertConfig,
    DistilBertTokenizerFast,
    get_linear_schedule_with_warmup,
)

print(f'Loading tokenizer: {PRETRAINED_MODEL}')
tokenizer = DistilBertTokenizerFast.from_pretrained(PRETRAINED_MODEL)

print(f'Loading model: {PRETRAINED_MODEL}')
config = DistilBertConfig.from_pretrained(
    PRETRAINED_MODEL,
    num_labels=NUM_LABELS,
    id2label={i: n for i, n in enumerate(LABEL_NAMES)},
    label2id={n: i for i, n in enumerate(LABEL_NAMES)},
    seq_classif_dropout=DROPOUT,
)
model = DistilBertForSequenceClassification.from_pretrained(
    PRETRAINED_MODEL,
    config=config,
    ignore_mismatched_sizes=True,
)
model.to(device)

# Freeze first FREEZE_LAYERS transformer layers
for i, layer in enumerate(model.distilbert.transformer.layer):
    if i < FREEZE_LAYERS:
        for param in layer.parameters():
            param.requires_grad = False

frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Frozen     : {frozen:,}  params (layers 0-{FREEZE_LAYERS-1})')
print(f'  Trainable  : {trainable:,}  params (layers {FREEZE_LAYERS}-5 + head)')

## STEP 7 — DataLoaders, Optimiser, Scheduler

In [ ]:
train_dataset = IntentDataset(df_train, tokenizer)
val_dataset   = IntentDataset(df_val,   tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE * 2,
                          shuffle=False, num_workers=2, pin_memory=True)

no_decay  = ['bias', 'LayerNorm.weight']
optimiser = torch.optim.AdamW(
    [
        {'params': [p for n, p in model.named_parameters()
                    if not any(nd in n for nd in no_decay)], 'weight_decay': WEIGHT_DECAY},
        {'params': [p for n, p in model.named_parameters()
                    if     any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
    ],
    lr=LEARNING_RATE,
)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimiser,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f'Steps per epoch : {len(train_loader)}')
print(f'Total steps     : {total_steps}')
print(f'Warmup steps    : {warmup_steps}')

## STEP 8 — Training Loop with Early Stopping

In [ ]:
import json
import torch.nn as nn
from tqdm.notebook import tqdm
from sklearn.metrics import f1_score, classification_report


def train_one_epoch(model, loader, optimiser, scheduler, device):
    loss_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    model.train()
    total_loss = 0.0
    bar = tqdm(loader, desc='  Training', leave=False)
    for batch in bar:
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        lbls  = batch['labels'].to(device)
        optimiser.zero_grad()
        loss  = loss_fn(model(input_ids=ids, attention_mask=mask).logits, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
        scheduler.step()
        total_loss += loss.item()
        bar.set_postfix({'loss': f'{loss.item():.4f}'})
    return total_loss / len(loader)


def evaluate(model, loader, device):
    loss_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='  Evaluating', leave=False):
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['labels'].to(device)
            out  = model(input_ids=ids, attention_mask=mask)
            total_loss += loss_fn(out.logits, lbls).item()
            all_preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())
    return (
        total_loss / len(loader),
        f1_score(all_labels, all_preds, average='macro'),
        all_preds,
        all_labels,
    )


# ── Training loop ─────────────────────────────────────────────────────────
history            = {'train_loss': [], 'val_loss': [], 'val_macro_f1': []}
best_val_f1        = 0.0
epochs_no_improve  = 0

print('=' * 60)
print('Starting training')
print(f'  LR={LEARNING_RATE}  smoothing={LABEL_SMOOTHING}  dropout={DROPOUT}')
print(f'  frozen_layers={FREEZE_LAYERS}  batch={BATCH_SIZE}')
print('=' * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    print(f'\nEpoch {epoch}/{NUM_EPOCHS}')
    print('-' * 40)

    train_loss                        = train_one_epoch(model, train_loader, optimiser, scheduler, device)
    val_loss, val_f1, val_preds, val_labels = evaluate(model, val_loader, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_macro_f1'].append(val_f1)

    print(f'  Train loss   : {train_loss:.4f}')
    print(f'  Val loss     : {val_loss:.4f}')
    print(f'  Val macro-F1 : {val_f1:.4f}')

    if val_f1 > best_val_f1:
        best_val_f1       = val_f1
        epochs_no_improve = 0
        model.save_pretrained(MODEL_SAVE_DIR)
        tokenizer.save_pretrained(MODEL_SAVE_DIR)
        print(f'  New best saved  (val macro-F1 = {best_val_f1:.4f})')
        with open(f'{RESULTS_DIR}/best_epoch_val_report.txt', 'w') as f:
            f.write(f'Best epoch: {epoch}\n\n')
            f.write(classification_report(val_labels, val_preds,
                                           target_names=LABEL_NAMES, digits=4))
    else:
        epochs_no_improve += 1
        print(f'  No improvement. Patience: {epochs_no_improve}/{EARLY_STOPPING_PATIENCE}')

    if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

with open(f'{RESULTS_DIR}/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nTraining complete. Best val macro-F1: {best_val_f1:.4f}')

## STEP 9 — Test Set Evaluation

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)
import matplotlib.pyplot as plt
import seaborn as sns

print('Loading best saved model for test evaluation...')
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
best_tok   = DistilBertTokenizerFast.from_pretrained(MODEL_SAVE_DIR)
best_model = DistilBertForSequenceClassification.from_pretrained(MODEL_SAVE_DIR)
best_model.to(device)
best_model.eval()

test_dataset = IntentDataset(df_test, best_tok)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2,
                           shuffle=False, num_workers=2, pin_memory=True)

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test inference'):
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        out  = best_model(input_ids=ids, attention_mask=mask)
        all_preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
        all_labels.extend(batch['labels'].numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

accuracy    = accuracy_score(all_labels, all_preds)
macro_f1    = f1_score(all_labels, all_preds, average='macro')
weighted_f1 = f1_score(all_labels, all_preds, average='weighted')
per_class   = f1_score(all_labels, all_preds, average=None)

print('\n' + '='*55)
print('TEST SET RESULTS  (real_test_simmc.csv)')
print('='*55)
print(f'  Accuracy    : {accuracy:.4f}  ({accuracy*100:.2f}%)')
print(f'  Macro-F1    : {macro_f1:.4f}')
print(f'  Weighted-F1 : {weighted_f1:.4f}')
print('\n  Per-class F1:')
for name, f1 in zip(LABEL_NAMES, per_class):
    bar = '#' * int(f1 * 40)
    print(f'    {name:<25} {f1:.4f}  {bar}')

report = classification_report(all_labels, all_preds,
                                target_names=LABEL_NAMES, digits=4)
print('\nFull classification report:')
print(report)

with open(f'{RESULTS_DIR}/test_classification_report.txt', 'w') as f:
    f.write('TEST SET — real_test_simmc.csv\n' + '='*55 + '\n\n' + report)

metrics = {
    'accuracy':     round(float(accuracy),    4),
    'macro_f1':     round(float(macro_f1),    4),
    'weighted_f1':  round(float(weighted_f1), 4),
    'per_class_f1': {n: round(float(f), 4) for n, f in zip(LABEL_NAMES, per_class)},
    'test_file':    'real_test_simmc.csv',
    'train_file':   'v5_train_mixed.csv',
    'total_test_rows': len(all_labels),
}
with open(f'{RESULTS_DIR}/test_metrics_summary.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Metrics saved to test_metrics_summary.json')

## STEP 10 — Confusion Matrix

In [ ]:
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            ax=ax, vmin=0, vmax=1)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix — v5 Mixed Real+Synthetic (row-normalised)', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png')

## STEP 11 — Training Curves

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_ran, history['train_loss'], 'b-o', label='Train Loss')
ax1.plot(epochs_ran, history['val_loss'],   'r-o', label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_ran, history['val_macro_f1'], 'g-o', label='Val Macro-F1')
best_epoch = history['val_macro_f1'].index(max(history['val_macro_f1'])) + 1
ax2.axvline(x=best_epoch, color='orange', linestyle='--',
            label=f'Best epoch ({best_epoch})')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Macro F1')
ax2.set_title('Validation Macro-F1 per Epoch')
ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('DistilBERT CRS — v5 Mixed Real+Synthetic Training', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

## STEP 12 — Per-Class F1 Bar Chart

In [ ]:
colors = ['#2196F3' if f >= 0.75 else '#FF9800' if f >= 0.60 else '#F44336'
          for f in per_class]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(LABEL_NAMES, per_class, color=colors, edgecolor='white', height=0.6)

for bar, val in zip(bars, per_class):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)

ax.axvline(x=macro_f1, color='black', linestyle='--',
           label=f'Macro-F1 = {macro_f1:.4f}')
ax.set_xlim(0, 1.1)
ax.set_xlabel('F1 Score', fontsize=12)
ax.set_title('Per-Class F1 Score — v5 Mixed Dataset Test Results', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: per_class_f1.png')

## STEP 13 — Quick Inference Demo

In [ ]:
def predict(history_turns, current_msg):
    parts = [f"{t['role'].upper()}: {t['content']}" for t in history_turns[-4:]]
    parts.append(f'CURRENT: {current_msg}')
    input_text = ' [SEP] '.join(parts)
    enc = best_tok(input_text, truncation=True, padding='max_length',
                   max_length=MAX_LEN, return_tensors='pt')
    with torch.no_grad():
        logits = best_model(enc['input_ids'].to(device),
                            enc['attention_mask'].to(device)).logits
    probs    = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    label_id = int(probs.argmax())
    return {
        'label':      LABEL_NAMES[label_id],
        'strategy':   RETRIEVAL_STRATEGY_MAP[label_id],
        'confidence': f'{probs[label_id]*100:.1f}%',
    }


hist = [
    {'role': 'user', 'content': 'I need a dress for a party'},
    {'role': 'bot',  'content': 'Option 1: Spring dress (black). Option 2: Carnival dress (red).'},
]

tests = [
    ([], 'I need a casual top for work',          'INITIAL_REQUEST'),
    (hist, 'need a cheaper one',                  'REFINEMENT'),
    (hist, 'is the first one machine washable?',  'ATTRIBUTE_QUESTION'),
    (hist, 'why did you recommend the black one?','EXPLANATION_WHY'),
    (hist, 'which one is better quality?',        'COMPARISON'),
    (hist, 'tell me more about the second one',   'SELECTION_REFERENCE'),
    (hist, "I'll take it",                        'FEEDBACK'),
    ([], 'hello',                                 'CHITCHAT'),
    (hist, 'I want a coat',                       'INITIAL_REQUEST'),  # mid-session switch
]

print(f'{"Test":<40} {"Predicted":<22} {"Expected":<22} {"OK?"}')
print('-' * 95)
correct = 0
for h, msg, expected in tests:
    r  = predict(h, msg)
    ok = 'OK' if r['label'] == expected else 'FAIL'
    if r['label'] == expected:
        correct += 1
    print(f'{msg:<40} {r["label"]:<22} {expected:<22} {ok}  {r["confidence"]}')

print(f'\nPassed: {correct}/{len(tests)}')

## STEP 14 — Download Trained Model and Results

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('distilbert_v5_outputs', 'zip', 'outputs')
print('Zipped outputs -> distilbert_v5_outputs.zip')
files.download('distilbert_v5_outputs.zip')
print('Download started.')

## After Downloading

1. Extract `distilbert_v5_outputs.zip`
2. Copy `best_model/` into your local `outputs/best_model/` (replacing the old one)
3. Your `predict.py` and `pipeline.py` will automatically use the new model

**Files saved in `outputs/results/`:**
- `test_classification_report.txt` — full per-class precision/recall/F1
- `test_metrics_summary.json` — accuracy, macro-F1, weighted-F1
- `confusion_matrix.png` — row-normalised confusion matrix
- `training_curves.png` — loss + F1 per epoch
- `per_class_f1.png` — per-class F1 bar chart
- `best_epoch_val_report.txt` — val report at best epoch
- `training_history.json` — full epoch-by-epoch metrics